### Imports

In [1]:
print("test")

test


In [2]:
import pandas as pd
import numpy as np

In [3]:
import pickle as pkl

In [4]:
df = pd.read_pickle("data/synthetic_data.pkl")

In [5]:
df.columns

Index(['district', 'size', 'rooms', 'price_per_sqm', 'price', 'altbau',
       'balkon', 'dachterrasse', 'erstbezug', 'garten', 'hohe_raeume',
       'klimaanlage', 'lift', 'loggia', 'terrasse', 'wintergarten', 'seller',
       'outside_area', 'id', 'title', 'anomaly_type'],
      dtype='str')

In [6]:
df['anomaly_label'] = [0 if x == '' else 1 for x in df['anomaly_type']]

In [7]:
#dropping columns for cleaner features:
"""
price - correlated with price_per_sqm
all variables making up outside area, e.g. balkon
title
id
anomaly_type
seller - IRL, I would keep it - private sellers tend to over/underprice, but I know we didn't inject anomalies based on that and our sample size to create the datset was too small for this to be statistically relevant.
title - created synthetically, might be insightful IRL
"""

features = df.loc[:,['district','size','rooms','price_per_sqm','outside_area','altbau','erstbezug', 'hohe_raeume','klimaanlage','lift','wintergarten']]

features.columns

Index(['district', 'size', 'rooms', 'price_per_sqm', 'outside_area', 'altbau',
       'erstbezug', 'hohe_raeume', 'klimaanlage', 'lift', 'wintergarten'],
      dtype='str')

### model evaluation function

In [37]:
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score
from datetime import datetime

In [38]:
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score

eval_df = pd.DataFrame(columns=['timestamp', 'model', 'comment', 'n_estimators', 'max_samples', 'accuracy', 'precision', 'recall', 'f1'])

def evaluate(y_true, y_pred, model_name, comment, n_estimators=None, max_samples=None, min_samples=None,eps=None):
    results = {
        'timestamp': datetime.now().strftime('%H:%M:%S'),
        'model': model_name,
        'comment': comment,
        'n_estimators': n_estimators, #for isolation forest
        'max_samples': max_samples, #for isolation forest
        'min_samples': min_samples, #for dbscan
        'eps': eps, #for dbscan
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred)
    }
    global eval_df
    eval_df = pd.concat([eval_df, pd.DataFrame([results])], ignore_index=True)
    return eval_df

### isolation forest

In [13]:
from sklearn.ensemble import IsolationForest

In [40]:
contamination = df['anomaly_label'].sum() / df['id'].count()

In [17]:
#features
features.head()

,district,size,rooms,price_per_sqm,outside_area,altbau,erstbezug,hohe_raeume,klimaanlage,lift,wintergarten
0,1,74,3.0,20407,0,0,0,0,0,0,0
1,1,101,3.5,20374,0,1,0,0,0,1,0
2,1,97,3.0,8315,0,0,1,0,0,0,0
3,1,195,5.5,17871,0,0,1,0,0,1,0
4,1,84,2.0,9695,0,0,0,0,0,1,0


In [41]:
#parameters
n_estimators = 100  # Number of trees
contamination = df['anomaly_label'].sum() / df['id'].count()  # Expected proportion of anomalies, ca 7.46%
sample_size = 256  # Number of samples used to train each tree

In [42]:
#fit model
iso_forest = IsolationForest(n_estimators=n_estimators,
                            contamination=contamination,
                            max_samples=sample_size,
                            random_state=42)
iso_forest.fit(features)

,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",256
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",np.float64(0....6521739130435)
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",None
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


In [43]:
y_pred = iso_forest.predict(features)

In [44]:
y_pred = [1 if x == -1 else 0 for x in y_pred]

In [45]:
y_true = df['anomaly_label']

In [47]:
model_name = "Isolation Forest"
comment = "Base model, no tuning"

evaluate(y_true, y_pred, model_name, comment, n_estimators=n_estimators, max_samples=sample_size)

,timestamp,model,comment,n_estimators,max_samples,accuracy,precision,recall,f1,min_samples,eps
0,18:28:37,Isolation Forest,"Base model, no tuning",None,None,0.895217,0.297376,0.297376,0.297376,None,None
1,18:30:09,Isolation Forest,"Base model, no tuning",None,None,0.895217,0.297376,0.297376,0.297376,None,None
2,18:32:03,Isolation Forest,"Base model, no tuning",100,256,0.895217,0.297376,0.297376,0.297376,None,None


In [48]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_true, y_pred))

[[4016  241]
 [ 241  102]]
